#  Loan Default Prediction — Machine Learning Exercises
**Week 4 | Day 1 — Fundamentals of Machine Learning**

---


##  Exercise 1 : Problem Definition & Data Collection Plan

---

###  Problem Statement

> **Goal:** Build a supervised binary classification model that predicts whether a loan applicant will **default** (fail to repay) or **not default** on a loan, based on applicant profile data and financial history.

Loan defaults represent a major financial risk for banks and lending institutions. By identifying high-risk applicants *before* disbursing funds, lenders can:
- Reduce non-performing assets (NPAs)
- Make better credit decisions
- Offer risk-adjusted interest rates

This is a **binary classification** problem:
- **Class 1 (Positive):** Applicant will default
- **Class 0 (Negative):** Applicant will repay the loan

---

###  Data Types Required

| Category | Features |
|---|---|
| **Personal / Demographic** | Age, gender, marital status, number of dependents, education level, employment type |
| **Financial Profile** | Monthly income, co-applicant income, existing debts, monthly expenses |
| **Credit History** | Credit score, past defaults, number of credit accounts, payment history |
| **Loan Details** | Loan amount requested, loan term (months), interest rate, loan purpose |
| **Property / Collateral** | Property area (urban/rural/semi-urban), property value, collateral type |
| **Behavioral Data** | Number of loan inquiries in the past 6 months, late payments in the last 2 years |

---

###  Data Sources

| Source | Description |
|---|---|
| **Internal Bank Records** | Application forms, existing loan repayment history, account transactions |
| **Credit Bureaus** | FICO scores, credit history reports (e.g., Equifax, Experian, TransUnion) |
| **Government / Public Data** | Tax records, employment verification databases |
| **Open Datasets** | Kaggle Loan Prediction datasets, LendingClub public data, UCI ML Repository |
| **Third-party APIs** | Employment verification services, income verification APIs |

---

###  Data Challenges to Anticipate

- **Class imbalance:** Defaults are typically rare (5–15% of applicants) → will require resampling techniques (SMOTE, class weights)
- **Missing values:** Income and credit history fields are frequently incomplete
- **Privacy/GDPR:** Personal data must be anonymized before use
- **Data drift:** Borrower behavior changes over time (e.g., during economic crises)


##  Exercise 2 : Feature Selection

---

### Dataset Loading & Exploration


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# --- Load the dataset ---
url = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%204/Day%201/Loan%20Predication.zip"
df = pd.read_csv(url)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()


In [ ]:
# --- Basic info ---
print("=== Dataset Info ===")
df.info()
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Target Distribution ===")
print(df['Loan_Status'].value_counts())


In [ ]:
# --- Visualize target distribution ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Target balance
df['Loan_Status'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','tomato'], edgecolor='black')
axes[0].set_title('Loan Status Distribution')
axes[0].set_xlabel('Status (Y = Approved, N = Rejected)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Approval rate by credit history
credit_approval = df.groupby('Credit_History')['Loan_Status'].apply(
    lambda x: (x == 'Y').mean()
).reset_index()
credit_approval.columns = ['Credit_History', 'Approval_Rate']
credit_approval['Credit_History'] = credit_approval['Credit_History'].map({0.0: 'Bad (0)', 1.0: 'Good (1)'})
axes[1].bar(credit_approval['Credit_History'], credit_approval['Approval_Rate'],
            color=['tomato', 'steelblue'], edgecolor='black')
axes[1].set_title('Loan Approval Rate by Credit History')
axes[1].set_ylabel('Approval Rate')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()


In [ ]:
# --- Correlation heatmap (numerical features) ---
numeric_df = df.copy()
numeric_df['Loan_Status_num'] = (df['Loan_Status'] == 'Y').astype(int)
numeric_df['Credit_History'] = pd.to_numeric(numeric_df['Credit_History'], errors='coerce')

corr_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
             'Loan_Amount_Term', 'Credit_History', 'Loan_Status_num']
corr = numeric_df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True)
plt.title('Feature Correlation with Loan Status')
plt.tight_layout()
plt.show()


In [ ]:
# --- Income distribution by loan status ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, col in zip(axes, ['ApplicantIncome', 'LoanAmount']):
    for status, color in [('Y', 'steelblue'), ('N', 'tomato')]:
        subset = df[df['Loan_Status'] == status][col].dropna()
        ax.hist(subset, bins=30, alpha=0.6, color=color,
                label=f'Status={status}', edgecolor='none')
    ax.set_title(f'{col} by Loan Status')
    ax.set_xlabel(col)
    ax.legend()

plt.tight_layout()
plt.show()


### ✅ Selected Features & Justification

Based on the exploratory analysis above, here are the most relevant features:

| Feature | Type | Justification |
|---|---|---|
| **Credit_History** | Binary | Strongest predictor (~correlation 0.54 with approval). Reflects past repayment behavior. |
| **LoanAmount** | Numerical | Higher amounts imply higher risk. Inversely correlated with approval. |
| **ApplicantIncome** | Numerical | Determines repayment capacity. Higher income → lower default risk. |
| **CoapplicantIncome** | Numerical | Additional income source increases repayment capacity. |
| **Loan_Amount_Term** | Numerical | Longer terms spread risk but may increase total exposure. |
| **Property_Area** | Categorical | Urban/semi-urban applicants tend to have better approval rates. |
| **Education** | Categorical | Graduate status correlates with higher income and lower default. |
| **Married** | Categorical | Dual-income households are generally more stable. |
| **Dependents** | Categorical | More dependents = higher expenses → higher default risk. |
| **Self_Employed** | Categorical | Self-employed income is less stable and harder to verify. |

**Features excluded:**
- `Loan_ID` → identifier, no predictive value
- `Gender` → low correlation and potential bias/legal concern


##  Exercise 3 : Model Selection, Training & Evaluation

---

###  Model Choices

For binary classification with tabular financial data, the best candidates are:

| Model | Pros | Cons |
|---|---|---|
| **Logistic Regression** | Interpretable, fast, good baseline | Assumes linear boundary, less powerful |
| **Random Forest** | Handles non-linearity, robust to outliers | Less interpretable |
| **Gradient Boosting (XGBoost)** | State-of-the-art on tabular data, handles imbalance | Slower to train, more hyperparameters |
| **Decision Tree** | Highly interpretable, good for stakeholder reports | Prone to overfitting |

We will train **Logistic Regression** and **Random Forest** and compare them.


In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, roc_auc_score, roc_curve,
                              classification_report)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# ---- Preprocessing ----
df_model = df.copy()

# Encode target
df_model['Loan_Status'] = (df_model['Loan_Status'] == 'Y').astype(int)

# Drop Loan_ID
df_model.drop(columns=['Loan_ID'], inplace=True)

# Encode categoricals
cat_cols = df_model.select_dtypes(include='object').columns
for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))

# Split features / target
X = df_model.drop(columns=['Loan_Status'])
y = df_model['Loan_Status']

# Impute missing values
imputer = SimpleImputer(strategy='most_frequent')
X_imputed = imputer.fit_transform(X)
X = pd.DataFrame(X_imputed, columns=X.columns)

# Train / test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
print(f"Target balance in test set:\n{y_test.value_counts(normalize=True).round(2)}")


In [ ]:
# ---- Train Models ----

# Logistic Regression (with scaling)
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42, max_iter=1000))
])
lr_pipeline.fit(X_train, y_train)

# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

print(" Models trained successfully.")


In [ ]:
# ---- Evaluation ----

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_test, y_pred):.4f}")
    print(f"  Recall    : {recall_score(y_test, y_pred):.4f}")
    print(f"  F1-Score  : {f1_score(y_test, y_pred):.4f}")
    print(f"  ROC-AUC   : {roc_auc_score(y_test, y_proba):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_test, y_pred, target_names=['Default', 'No Default']))
    return y_pred, y_proba

lr_pred, lr_proba = evaluate_model("Logistic Regression", lr_pipeline, X_test, y_test)
rf_pred, rf_proba = evaluate_model("Random Forest", rf_model, X_test, y_test)


In [ ]:
# ---- Confusion Matrices & ROC Curve ----

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrices
for ax, preds, name in zip(axes[:2],
                            [lr_pred, rf_pred],
                            ['Logistic Regression', 'Random Forest']):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Default', 'No Default'],
                yticklabels=['Default', 'No Default'])
    ax.set_title(f'Confusion Matrix\n{name}')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

# ROC Curves
ax = axes[2]
for proba, name, color in [(lr_proba, 'Logistic Regression', 'steelblue'),
                             (rf_proba, 'Random Forest', 'tomato')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, color=color, label=f'{name} (AUC={auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_title('ROC Curves')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ---- Cross-Validation ----

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in [('Logistic Regression', lr_pipeline), ('Random Forest', rf_model)]:
    scores = cross_val_score(model, X, y, cv=cv, scoring='f1')
    print(f"{name:25s} | CV F1: {scores.mean():.4f} ± {scores.std():.4f}")


In [ ]:
# ---- Feature Importance (Random Forest) ----

importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values(ascending=True)

plt.figure(figsize=(9, 6))
importances_sorted.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()


###  Metrics Explained

| Metric | Why it matters for Loan Default |
|---|---|
| **Accuracy** | Overall correctness, but misleading with imbalanced classes |
| **Precision** | Of all predicted defaults, how many actually defaulted? (Avoids false alarms) |
| **Recall** | Of all actual defaults, how many did we catch? (**Most critical** — missing a default is costly) |
| **F1-Score** | Harmonic mean of Precision & Recall — best single metric for imbalanced data |
| **ROC-AUC** | Model's ability to distinguish between classes across all thresholds |
| **Cross-Validation** | Ensures results generalize and aren't due to lucky train/test split |

>  In loan default prediction, **Recall is the priority**: missing a default (false negative) is far more costly than a false alarm (false positive).


## 🌟 Exercise 4 : ML Types for Specific Problems

---

### Scenario 1 —  Predicting Stock Prices

**Type: Supervised Learning — Regression**

Stock price prediction involves learning from historical labeled data (past prices, volumes, indicators) to predict a **continuous numerical output** (future price).

- Input features: past prices, trading volume, moving averages, macroeconomic indicators
- Output: future stock price (continuous value)
- Suitable models: LSTM (time series), XGBoost Regressor, ARIMA

> Why not classification? Although one could predict "up/down", predicting the actual price value is a regression task. Why not unsupervised? We have labeled historical data (price at time T is the label for features at T-1).

---

### Scenario 2 — 📚 Organizing a Library of Books

**Type: Unsupervised Learning — Clustering**

There are no predefined genre labels. The goal is to **discover natural groupings** in the data based on similarities in content, style, vocabulary, or metadata.

- Input: book descriptions, text features (TF-IDF, embeddings), author info
- Output: clusters = emergent genre groups
- Suitable models: K-Means, DBSCAN, Hierarchical Clustering, LDA (topic modeling)

> Why not supervised? We have no labeled training data. Why not RL? There is no agent making sequential decisions or receiving environment feedback.

---

### Scenario 3 —  Robot in a Maze (Shortest Path)

**Type: Reinforcement Learning**

The robot must **learn through trial and error** by interacting with the maze environment. It receives rewards for progress and penalties for hitting walls or taking longer paths.

- Agent: the robot
- Environment: the maze
- Actions: move up/down/left/right
- Reward: +1 for reaching the exit, -0.01 per step (encourages efficiency)
- Suitable algorithms: Q-Learning, Deep Q-Network (DQN)

> Why not supervised? There is no labeled dataset of "correct paths". Why not unsupervised? There is a clear objective (reaching the goal) that drives learning through reward signals.

---

###  Summary Table

| Scenario | ML Type | Paradigm | Key Reason |
|---|---|---|---|
| Stock Price Prediction | Supervised – Regression | Labeled data, continuous output | Historical prices are labeled |
| Book Library Organization | Unsupervised – Clustering | No labels, find structure | No predefined categories |
| Robot Maze Navigation | Reinforcement Learning | Agent + environment + reward | Sequential decisions, trial & error |


## 🌟 Exercise 5 : Evaluation Strategies for Different ML Models

---

### 1. 🎯 Supervised Learning — Classification (Logistic Regression / Random Forest)

**Evaluation Strategy:**

| Method | Description |
|---|---|
| **Train/Test Split** | Hold out 20% of data; train on 80%. Quick baseline evaluation. |
| **Stratified K-Fold CV** | 5 or 10 folds preserving class ratio. More reliable than single split. |
| **Accuracy** | % of correct predictions. Misleading with imbalanced classes. |
| **Precision & Recall** | Precision: quality of positive predictions. Recall: coverage of actual positives. |
| **F1-Score** | Harmonic mean — best single metric when classes are imbalanced. |
| **ROC-AUC** | Area under ROC curve. Measures discrimination ability across thresholds. |
| **Confusion Matrix** | Reveals FP, FN, TP, TN breakdown visually. |
| **Calibration Curve** | Checks if predicted probabilities are reliable. |

**Challenges:**
- Class imbalance inflates accuracy (a model predicting "no default" always gets 85%+ accuracy)
- Data leakage can make models appear better than they are
- Need to choose the right decision threshold for business needs


In [ ]:
# --- Supervised: Demonstrate Cross-Validation + ROC ---
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
fpr, tpr, _ = roc_curve(y_test, rf_proba)
auc = roc_auc_score(y_test, rf_proba)
axes[0].plot(fpr, tpr, color='tomato', lw=2, label=f'Random Forest (AUC={auc:.3f})')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='tomato')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random Classifier')
axes[0].set_title('ROC Curve — Supervised Model')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

# Calibration
prob_true, prob_pred = calibration_curve(y_test, rf_proba, n_bins=8)
axes[1].plot(prob_pred, prob_true, 's-', color='steelblue', label='Random Forest')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfect calibration')
axes[1].set_title('Calibration Curve — Supervised Model')
axes[1].set_xlabel('Mean Predicted Probability')
axes[1].set_ylabel('Fraction of Positives')
axes[1].legend()

plt.tight_layout()
plt.show()

# CV Scores
cv_scores = cross_val_score(rf_model, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring='roc_auc')
print(f"Random Forest | 5-Fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


---
### 2. 🔵 Unsupervised Learning — Clustering (K-Means)

**Evaluation Strategy:**

| Method | Description |
|---|---|
| **Elbow Method** | Plot inertia vs K; look for the "elbow" where adding clusters gives diminishing returns |
| **Silhouette Score** | Measures cohesion vs separation (-1 to 1). Higher = better-defined clusters |
| **Davies-Bouldin Index** | Average ratio of within-cluster scatter to between-cluster separation. Lower = better |
| **Cluster Profiling** | Examine mean feature values per cluster to validate interpretability |
| **Visualization (PCA/t-SNE)** | Reduce to 2D and plot clusters visually |

**Challenges:**
- No ground truth labels → no "correct answer" to compare against
- Results depend heavily on K choice and feature scaling
- Silhouette score can be misleading with non-convex or unequal-size clusters
- High-dimensional data requires dimensionality reduction before clustering


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

# Use numerical features for clustering demo
cluster_features = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
X_cluster = X[cluster_features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# --- Elbow Method ---
inertias, silhouettes = [], []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Elbow
axes[0].plot(K_range, inertias, 'bo-', lw=2)
axes[0].set_title('Elbow Method (Inertia)')
axes[0].set_xlabel('Number of Clusters K')
axes[0].set_ylabel('Inertia')
axes[0].axvline(x=3, color='tomato', linestyle='--', alpha=0.7, label='Optimal K=3')
axes[0].legend()

# Silhouette
axes[1].plot(K_range, silhouettes, 'rs-', lw=2)
axes[1].set_title('Silhouette Score vs K')
axes[1].set_xlabel('Number of Clusters K')
axes[1].set_ylabel('Silhouette Score')
best_k = K_range[silhouettes.index(max(silhouettes))]
axes[1].axvline(x=best_k, color='steelblue', linestyle='--', alpha=0.7, label=f'Best K={best_k}')
axes[1].legend()

# PCA 2D Cluster Visualization
km_final = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(X_scaled)
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)
scatter = axes[2].scatter(X_2d[:, 0], X_2d[:, 1], c=cluster_labels, cmap='viridis', alpha=0.5, s=15)
axes[2].set_title('K-Means Clusters (PCA 2D)')
axes[2].set_xlabel('PC1'); axes[2].set_ylabel('PC2')
plt.colorbar(scatter, ax=axes[2], label='Cluster')

plt.tight_layout()
plt.show()

# Print scores
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
labels3 = km3.fit_predict(X_scaled)
print(f"K=3 | Silhouette Score    : {silhouette_score(X_scaled, labels3):.4f}  (closer to 1 = better)")
print(f"K=3 | Davies-Bouldin Index: {davies_bouldin_score(X_scaled, labels3):.4f}  (closer to 0 = better)")


---
### 3. 🕹️ Reinforcement Learning — Maze Navigation (Q-Learning)

**Evaluation Strategy:**

| Method | Description |
|---|---|
| **Cumulative Reward** | Total reward per episode over time. Should trend upward as agent improves. |
| **Convergence** | Q-table stabilizes → policy no longer changes significantly between episodes |
| **Steps to Goal** | Number of steps taken to reach the exit per episode. Should decrease over time. |
| **Success Rate** | % of episodes where the agent reaches the goal within a time limit |
| **Exploration vs Exploitation (ε-decay)** | Monitor ε over time to ensure the agent transitions from exploring to exploiting |
| **Policy Visualization** | Display the learned path in the maze to validate it's the shortest/optimal route |

**Challenges:**
- Sparse rewards: the agent may rarely reach the goal early on → slow learning
- Reward shaping: poorly designed rewards can produce unintended behaviors
- Convergence is not guaranteed in large/continuous state spaces
- Evaluation is highly environment-specific — performance doesn't transfer across mazes

> **Note:** A complete RL implementation requires a maze environment (e.g., OpenAI Gym or custom). Below is a conceptual simulation of the key metrics.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Simulated RL Training Metrics ----
np.random.seed(42)
n_episodes = 500

# Simulate cumulative reward curve (starts noisy, improves over time)
rewards = -50 + np.cumsum(np.random.normal(0.15, 0.5, n_episodes))
rewards = np.clip(rewards, -60, 0)  # reward asymptotes near 0 (optimal)

# Simulate steps per episode (starts high, decreases as agent learns)
steps = 80 * np.exp(-np.linspace(0, 3, n_episodes)) + 10 + np.random.normal(0, 3, n_episodes)
steps = np.clip(steps, 8, 100)

# Simulate epsilon decay
epsilon = np.exp(-np.linspace(0, 5, n_episodes)) * 0.9 + 0.05

# ---- Plot ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(rewards, color='steelblue', alpha=0.7, lw=1.5)
axes[0].set_title('Cumulative Reward per Episode')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Cumulative Reward')
axes[0].axhline(-10, color='tomato', linestyle='--', alpha=0.7, label='Near-optimal threshold')
axes[0].legend()

# Smooth steps with rolling average
window = 20
steps_smooth = pd.Series(steps).rolling(window).mean()
axes[1].plot(steps, color='lightgray', alpha=0.5, lw=1)
axes[1].plot(steps_smooth, color='tomato', lw=2, label=f'Rolling avg ({window} ep)')
axes[1].set_title('Steps to Goal per Episode')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Steps')
axes[1].legend()

axes[2].plot(epsilon, color='green', lw=2)
axes[2].fill_between(range(n_episodes), epsilon, alpha=0.1, color='green')
axes[2].set_title('Epsilon (Exploration Rate) Decay')
axes[2].set_xlabel('Episode'); axes[2].set_ylabel('ε (epsilon)')
axes[2].axhline(0.05, color='tomato', linestyle='--', alpha=0.7, label='Min ε = 0.05')
axes[2].legend()

plt.tight_layout()
plt.show()

print("RL Training Summary (simulated):")
print(f"  Initial avg steps   : {steps[:20].mean():.1f}")
print(f"  Final avg steps     : {steps[-20:].mean():.1f}  ← agent learned a shorter path")
print(f"  Final epsilon       : {epsilon[-1]:.3f}  ← mostly exploiting, some exploring")


---

## 📌 Final Summary — Evaluation Across ML Paradigms

| Aspect | Supervised (Classification) | Unsupervised (Clustering) | Reinforcement Learning |
|---|---|---|---|
| **Ground Truth** | ✅ Available (labels) | ❌ Not available | ⚠️ Defined by reward |
| **Primary Metrics** | Accuracy, F1, AUC-ROC | Silhouette, Elbow, DB Index | Cumulative reward, convergence |
| **Validation Method** | K-Fold CV, ROC curve | Internal indices + visualization | Episode-based performance tracking |
| **Key Challenge** | Class imbalance, overfitting | Choosing K, interpretability | Sparse rewards, exploration tradeoff |
| **Output** | Class probabilities | Cluster assignments | Learned policy / Q-table |

---
*Notebook completed — Week 4 | Day 1 | Machine Learning Fundamentals*
